In [8]:
pip install openai

Looking in indexes: https://nexus.raisa.go.rshbank.ru/repository/pypi/simple
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import date, datetime
from typing import Any

import json
import math
import time

import numpy as np
import pandas as pd
import httpx

from openai import (
    OpenAI,
    RateLimitError,
    APIStatusError,
    APIConnectionError,
    APITimeoutError,
)

from IPython.display import display, Markdown

import csv, hashlib, traceback, re, html
from tqdm.auto import tqdm
from IPython.display import HTML


In [ ]:

INPUT_FOLDER = Path("./input")
OUTPUT_FOLDER = Path("./output")

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = 'CodegenChatReasoningUltra/codegen'


PROCESS_ANALYSIS_PROMPT = """
Ты — эксперт по анализу рабочего времени работников, оцифровке операций с использованием технологии Task Mining с практическим опытом повышения эффективности труда работников, оптимизации, анализа операций.

Задача: проанализировать активность работника за компьютером в течение 1 дня, определить рутинные цепочки операций, выявить нецелевые действия в системах, определить зоны оптимизации с точки зрения исключения дублирования функционала, поиска операций под роботизацию, автоматизацию, определить особенности труда работника.

Информация о работнике:

отдел учета и операционного сопровождения кредитов юридических лиц и гарантий, Единый сервисный центр "Кредитный";

основной функционал: проведение операций в АБС ЦФТ в части сопровождения кредитов юридических лиц и гарантий;

стандартный рабочий график 5/2 с 09:00 до 18:00 с понедельника по четверг и с 09:00 до 16:45 по пятницам 

должность: главный специалист

работник использует несколько рабочих станций (поле "Номер машины")

Для анализа ты используешь 2 файла:

activity (данные агента мониторинга Task Mining). Файл состоит из полей:
- action (подробное описание действия работника с указанием ПО, в котором было совершено действие и указанием нажатых клавиш
- worker_id (уникальный идентификатор работника)
- ms_time (длительность, прошедшая с начала эпохи (01.01.1970 00:00:00.000 UTC) до момента совершения действия)
- computer_name (идентификатор компьютера, может быть как физической машиной, так и виртуальной (VDI, терминальная сессия))

cfg_op_short (пример конфигурационного файла, который используется для ручной разметки операций работника по его активности за компьютером). Состав полей файла:
- action_c (название действия в рамках операции)
- starter_c (признак того, является ли данной действие начальным: 1- да, 0 - нет)
- finisher_c (признак того, является ли данной действие конечным: 1- да, 0 - нет)

При анализе активности сотрудника используй файл cfg_op_short в качестве образца для определения цепочек операций. Правила разметки, приведённые в cfg_op_short, могут оказаться неприменимыми к той активности, которая рассматривается в данном запросе. Твоя задача - обнаружить новые цепочки операций, опираясь на правила из cfg_op_short как на пример, а не пытаться находить операции исключительно по действиям, перечисленным в полях starter_c, finisher_c.

Методология анализа (которой ты должен придерживаться):
При анализе используй лучшие практики:
A. Анализ структуры операций
B. Анализ времени
C. Анализ отклонений
D. Анализ эффективности
E. Корневые причины
F. Выявление основного функционала и стандартной рабочей рутины (например, просмотр почты)

Структурируй ответ в формате таблицы со следующими полями:
1. Выявленная неоптимальная цепочка действий / действие - укажи примеры из таблицы activity из поля action
2. Описание проблемы в 1-2 предложениях.
3. Предложение быстрого устранения проблемы (должно содержать в себе конкретное действие), если применимо к инсайту
4. Стратегическое воздействие для устранения проблемы (не quick win, а более трудоемкое и длительное мероприятие)
5. Оценка потенциального эффекта (включая формулы расчета потенциального эффекта (высвобождение трудозатрат, сокращение длительности, повышение производительности)
У тебя нет ограничений на количество выявленных инсайтов, укажи все, что найдешь и отранжируй по потенциальному эффекту.

Дополнительно выведи краткое связное описание рабочего дня сотрудника объемом 3–5 предложений. Текст должен отвечать на вопрос: «Чем занимался сотрудник в течение рабочего дня?».

Принципы анализа
- Не описывай метрики, а интерпретируй их.
- Не повторяй входные данные.
- Фокусируйся на причинно-следственных связях.
- Отделяй симптомы от первопричин.
- Предлагай конкретные действия, а не общие рекомендации.
- Приоритизируй выводы по бизнес-влиянию.
- Подтверждай свои выводы фактами из входных данных с конкретными цифрами.
- Подкрепляй к своим выводам примеры из исходного лога (цепочки действий и метку в миллисекундах, когда произошло первое действие), а не из файла с правилами.
"""

MAX_DATA_CHARS = 20_000
MAX_CONSOLIDATION_CHARS = 35_000

ANALYSIS_MAX_TOKENS = 4_500
CONSOLIDATION_MAX_TOKENS = 5_000
FINAL_MAX_TOKENS = 7_000

MAX_RETRIES = 10
REQUEST_DELAY_SECONDS = 20
RATE_LIMIT_WAIT_SECONDS = 65

SHOW_PROGRESS = True


# Диагностика и возобновление (не изменяют методологию анализа)
CACHE_FOLDER = OUTPUT_FOLDER / "tm_cache"
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)
PERFORMANCE_LOG = OUTPUT_FOLDER / "performance_log.csv"
ENABLE_CACHE = True
# При изменении промптов, модели и данных ключ кеша изменится автоматически.


In [4]:
llm_client = OpenAI(
    base_url ='123',
    api_key='123',
    http_client=httpx.Client(verify=False)
)
llm_client.models.list()

SyncPage[Model](data=[Model(id='CodegenAutocompletion/codegen', created=1677610602, object='model', owned_by='openai'), Model(id='CodegenChat/codegen', created=1677610602, object='model', owned_by='openai'), Model(id='CodegenChatReasoning/codegen', created=1677610602, object='model', owned_by='openai'), Model(id='CodegenEmbedding/codegen', created=1677610602, object='model', owned_by='openai'), Model(id='CodegenChatReasoningUltra/codegen', created=1677610602, object='model', owned_by='openai'), Model(id='CodegenChatReasoningDeprecated/codegen', created=1677610602, object='model', owned_by='openai')], object='list')

In [ ]:
def log(message: str) -> None:
    if SHOW_PROGRESS:
        print(message)


def normalize_value(value: Any) -> Any:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        normalized = float(value)

        if math.isnan(normalized) or math.isinf(normalized):
            return None

        return normalized

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (str, int, float, bool)):
        return value

    return str(value)


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.dropna(
        axis=0,
        how="all",
    )

    df = df.dropna(
        axis=1,
        how="all",
    )

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    return df.reset_index(drop=True)


def dataframe_part_to_json(
    df: pd.DataFrame,
    start_row: int,
) -> str:
    rows = []

    for row in df.itertuples(
        index=False,
        name=None,
    ):
        rows.append([
            normalize_value(value)
            for value in row
        ])

    payload = {
        "columns": [
            str(column)
            for column in df.columns
        ],
        "row_start": start_row + 1,
        "row_end": start_row + len(df),
        "rows": rows,
    }

    return json.dumps(
        payload,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def split_dataframe_by_size(
    df: pd.DataFrame,
    max_chars: int = MAX_DATA_CHARS,
) -> list[dict[str, Any]]:
    if df.empty:
        return []

    chunks = []

    start = 0
    total_rows = len(df)

    while start < total_rows:
        low = start + 1
        high = total_rows

        best_end = None
        best_json = None

        while low <= high:
            middle = (low + high) // 2

            candidate_df = df.iloc[start:middle]

            candidate_json = dataframe_part_to_json(
                candidate_df,
                start_row=start,
            )

            if len(candidate_json) <= max_chars:
                best_end = middle
                best_json = candidate_json
                low = middle + 1
            else:
                high = middle - 1

        if best_end is None:
            best_end = start + 1

            best_json = dataframe_part_to_json(
                df.iloc[start:best_end],
                start_row=start,
            )

        chunks.append({
            "start_row": start + 1,
            "end_row": best_end,
            "json": best_json,
        })

        start = best_end

    return chunks


def split_text_by_size(
    text: str,
    max_chars: int = MAX_CONSOLIDATION_CHARS,
) -> list[str]:
    text = text.strip()

    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    sections = text.split("\n\n")

    chunks = []
    current_parts = []
    current_size = 0

    for section in sections:
        section_size = len(section) + 2

        if (
            current_parts
            and current_size + section_size > max_chars
        ):
            chunks.append(
                "\n\n".join(current_parts)
            )

            current_parts = []
            current_size = 0

        if section_size > max_chars:
            if current_parts:
                chunks.append(
                    "\n\n".join(current_parts)
                )

                current_parts = []
                current_size = 0

            for position in range(
                0,
                len(section),
                max_chars,
            ):
                chunks.append(
                    section[
                        position:
                        position + max_chars
                    ]
                )

            continue

        current_parts.append(section)
        current_size += section_size

    if current_parts:
        chunks.append(
            "\n\n".join(current_parts)
        )

    return chunks

# Журнал записывается сразу после каждой попытки и паузы.
PERF_FIELDS = ["timestamp", "stage", "part", "attempt", "status", "seconds", "prompt_tokens", "completion_tokens", "reasoning_tokens", "total_tokens", "finish_reason", "wait_seconds", "error"]

def perf_event(**values):
    PERFORMANCE_LOG.parent.mkdir(parents=True, exist_ok=True)
    row = {key: values.get(key, "") for key in PERF_FIELDS}
    row["timestamp"] = datetime.now().isoformat(timespec="seconds")
    with PERFORMANCE_LOG.open("a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=PERF_FIELDS)
        if f.tell() == 0:
            writer.writeheader()
        writer.writerow(row)
        f.flush()

def cache_call(stage, part, system_prompt, user_prompt, max_tokens):
    key_data = json.dumps([MODEL_NAME, stage, system_prompt, user_prompt, max_tokens], ensure_ascii=False)
    digest = hashlib.sha256(key_data.encode("utf-8")).hexdigest()[:20]
    safe_stage = re.sub(r"[^a-zA-Z0-9_-]", "_", stage)
    cache_path = CACHE_FOLDER / f"{safe_stage}_{digest}.md"
    if ENABLE_CACHE and cache_path.exists():
        result = cache_path.read_text(encoding="utf-8")
        if result.strip():
            log(f"  ♻ Кеш: {stage} {part}")
            perf_event(stage=stage, part=part, status="cache_hit", seconds=0)
            return result
    result = ask_model(system_prompt, user_prompt, max_tokens, stage=stage, part=part)
    if ENABLE_CACHE:
        cache_path.write_text(result, encoding="utf-8")
    return result

def render_html(markdown_text):
    try:
        import markdown as md
        body = md.markdown(markdown_text, extensions=["tables", "fenced_code"])
    except ImportError:
        # Без дополнительной зависимости отчёт остаётся читаемым.
        body = "<pre>" + html.escape(markdown_text) + "</pre>"
    return """<!doctype html><html lang="ru"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Анализ Task Mining</title><style>
    body{background:#f2f7f3;color:#183527;font:16px/1.65 system-ui,Arial,sans-serif;margin:0;padding:32px}
    main{max-width:1300px;margin:auto;background:white;padding:36px;border-radius:18px;box-shadow:0 8px 35px #143d2414}
    h1,h2,h3{color:#12643b}h1{border-bottom:3px solid #42a86c;padding-bottom:15px}
    table{border-collapse:collapse;display:block;overflow-x:auto;width:100%;margin:18px 0}
    th,td{border:1px solid #d7e6db;padding:12px;text-align:left;vertical-align:top;min-width:130px}
    th{background:#e3f3e8;color:#124d2d}tr:nth-child(even){background:#f8fbf9}
    pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f8f6;padding:18px;border-radius:10px}
    blockquote{border-left:4px solid #4b9d68;padding-left:16px;color:#456653}
    </style></head><body><main>""" + body + "</main></body></html>"


In [ ]:
def ask_model(system_prompt: str, user_prompt: str, max_tokens: int, stage="analysis", part="") -> str:
    current_max_tokens = max_tokens
    empty_content_attempts = 0
    for attempt in range(1, MAX_RETRIES + 1):
        started = time.perf_counter()
        try:
            response = llm_client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=current_max_tokens,
                extra_body={"chat_template_kwargs": {"enable_thinking": True}},
            )
            elapsed = time.perf_counter() - started
            choice = response.choices[0]
            content = (choice.message.content or "").strip()
            usage = response.usage
            input_tokens = getattr(usage, "prompt_tokens", None)
            output_tokens = getattr(usage, "completion_tokens", None)
            total_tokens = getattr(usage, "total_tokens", None)
            details = getattr(usage, "completion_tokens_details", None)
            reasoning_tokens = getattr(details, "reasoning_tokens", None)
            finish = choice.finish_reason
            status = "success" if content and finish != "length" else ("truncated" if finish == "length" else "empty")
            perf_event(stage=stage, part=part, attempt=attempt, status=status,
                       seconds=round(elapsed, 2), prompt_tokens=input_tokens,
                       completion_tokens=output_tokens, reasoning_tokens=reasoning_tokens,
                       total_tokens=total_tokens, finish_reason=finish)
            log(f"  [{stage} {part}] {elapsed:.1f} с | input={input_tokens} output={output_tokens} "
                f"reasoning={reasoning_tokens} finish={finish} | {status}")
            if finish == "length":
                raise RuntimeError(f"ОТВЕТ ОБРЕЗАН: {stage} {part}; выходных токенов {output_tokens}, "
                                   f"лимит max_tokens={current_max_tokens}. Сократите вход или измените лимит.")
            if content:
                return content
            empty_content_attempts += 1
            if empty_content_attempts >= 3:
                raise RuntimeError(f"Модель {stage} {part} не вернула итоговый текст после 3 попыток")
            current_max_tokens = min(current_max_tokens + 2000, 12000)
            log(f"  Пустой ответ, увеличен max_tokens до {current_max_tokens}")
            perf_event(stage=stage, part=part, attempt=attempt, status="empty_wait", wait_seconds=10)
            time.sleep(10)
        except RateLimitError as exc:
            elapsed = time.perf_counter() - started
            wait = RATE_LIMIT_WAIT_SECONDS
            perf_event(stage=stage, part=part, attempt=attempt, status="rate_limit_429", seconds=round(elapsed,2),
                       wait_seconds=wait if attempt < MAX_RETRIES else 0, error=str(exc)[:1000])
            log(f"  HTTP 429: {exc}; попытка {attempt}/{MAX_RETRIES}")
            if attempt >= MAX_RETRIES: raise
            time.sleep(wait)
        except (APIConnectionError, APITimeoutError) as exc:
            elapsed = time.perf_counter() - started
            wait = min(10 * attempt, 60)
            perf_event(stage=stage, part=part, attempt=attempt, status=type(exc).__name__, seconds=round(elapsed,2),
                       wait_seconds=wait if attempt < MAX_RETRIES else 0, error=str(exc)[:1000])
            log(f"  {type(exc).__name__}: {exc}; попытка {attempt}/{MAX_RETRIES}")
            if attempt >= MAX_RETRIES: raise
            time.sleep(wait)
        except APIStatusError as exc:
            elapsed = time.perf_counter() - started
            wait = min(10 * attempt, 60) if exc.status_code >= 500 and attempt < MAX_RETRIES else 0
            perf_event(stage=stage, part=part, attempt=attempt, status=f"http_{exc.status_code}",
                       seconds=round(elapsed,2), wait_seconds=wait, error=str(exc)[:1500])
            log(f"  HTTP {exc.status_code}: {exc}")
            if not wait: raise
            time.sleep(wait)
    raise RuntimeError("Не удалось получить ответ модели")


In [ ]:
def analyze_data_chunk(
    file_name: str,
    chunk_json: str,
    part_number: int,
    total_parts: int,
    start_row: int,
    end_row: int,
) -> str:
    system_prompt = """
Ты выполняешь анализ активности работника.

Используй углубленное reasoning, но возвращай только компактные,
фактически подтвержденные наблюдения для дальнейшего объединения.

Не формируй отдельный итоговый отчет по файлу.
Не пересказывай таблицу.
Не придумывай отсутствующие факты.
"""

    user_prompt = f"""
Все входные Excel-файлы относятся к дню активности работника.

Текущий источник данных:
{file_name}

Часть:
{part_number} из {total_parts}

Строки:
{start_row}–{end_row}

Итоговая задача анализа:

{PROCESS_ANALYSIS_PROMPT}

Проанализируй текущую часть как один из источников общей активности работника за 1 день.

Верни компактный промежуточный материал:

1. Значимые наблюдения.
2. Числовые подтверждения.
3. Возможные связи с другими типами метрик.
4. Возможные проблемы и отклонения.
5. Гипотезы о причинах с явным указанием, что это гипотезы.
6. Название файла-источника.

Не составляй финальный отчет.
Не добавляй формальное вступление и заключение.

ДАННЫЕ:

{chunk_json}
"""

    return cache_call("analysis", f"{file_name}:{part_number}/{total_parts}", system_prompt, user_prompt, ANALYSIS_MAX_TOKENS)




def consolidate_process_materials(
    intermediate_results: list[str],
) -> str:
    current_text = "\n\n".join(
        (
            f"===== ПРОМЕЖУТОЧНЫЙ МАТЕРИАЛ {index} =====\n\n"
            f"{result}"
        )
        for index, result in enumerate(
            intermediate_results,
            start=1,
        )
    )

    level = 1

    while len(current_text) > MAX_CONSOLIDATION_CHARS:
        blocks = split_text_by_size(
            current_text,
            MAX_CONSOLIDATION_CHARS,
        )

        log(
            f"Уровень консолидации {level}: "
            f"{len(blocks)} блоков."
        )

        consolidated_blocks = []

        for block_number, block in enumerate(
            blocks,
            start=1,
        ):
            system_prompt = """
Ты объединяешь промежуточные результаты выявления неоптимальных действий работника из частей одного файла.

Используй reasoning для перекрестной проверки наблюдений,
устранения дублей и выявления связей между разными метриками.

Верни только компактный материал для следующего этапа.
Не формируй пока окончательный отчет пользователю.
"""

            user_prompt = f"""
Основная задача анализа:

{PROCESS_ANALYSIS_PROMPT}

Ниже представлены промежуточные наблюдения,
полученные из разных Excel-файлов и частей файлов.

Выполни перекрестную консолидацию:

1. Объедини одинаковые и связанные выводы.
2. Удали дубли.
3. Сохрани важные числовые показатели.
4. Сохрани названия файлов-источников.
5. Отдели устойчивые закономерности от единичных случаев.
6. Явно обозначь противоречия между источниками.
7. Не придумывай отсутствующие факты.
8. Не формируй отдельные разделы по каждому файлу.

Текущий блок:
{block_number} из {len(blocks)}

МАТЕРИАЛЫ:

{block}
"""

            result = cache_call("merge", f"level{level}:block{block_number}/{len(blocks)}", system_prompt, user_prompt, CONSOLIDATION_MAX_TOKENS)

            consolidated_blocks.append(result)

            time.sleep(
                REQUEST_DELAY_SECONDS
            )

        previous_length = len(current_text)
        current_text = "\n\n".join(
            (
                f"===== КОНСОЛИДИРОВАННЫЙ БЛОК {index} =====\n\n"
                f"{result}"
            )
            for index, result in enumerate(
                consolidated_blocks,
                start=1,
            )
        )

        if len(current_text) >= previous_length and len(current_text) > MAX_CONSOLIDATION_CHARS:
            raise RuntimeError("Консолидация не сокращает материал; уменьшите объём промежуточных ответов")
        level += 1

    return current_text


# ============================================================
# ЕДИНЫЙ ИТОГОВЫЙ ОТЧЕТ
# ============================================================

def create_final_process_report(
    consolidated_material: str,
    files_info: list[dict[str, Any]],
) -> str:
    sources_description = "\n".join(
        (
            f"- {item['file_name']}: "
            f"{item['rows']} строк, "
            f"{item['columns']} столбцов"
        )
        for item in files_info
    )

    system_prompt = """
Ты ведущий эксперт по Task Mining.

Используй reasoning для комплексного анализа взаимосвязей
между всеми выявленными цепочками действий.

Верни только единый итоговый отчет по активности работника.
Формат ответа - строго Markdown.
Используй только стандартную Markdown-разметку
Не раскрывай внутренний ход рассуждений.
Не описывай техническую обработку файлов.
Не формируй отдельные отчеты по файлам.
"""

    user_prompt = f"""
Сформируй единый консолидированный анализ активности работника.

Основная задача и требуемый формат:

{PROCESS_ANALYSIS_PROMPT}

Использованные источники:

{sources_description}

Ниже приведен объединенный аналитический материал,
полученный из всех входных Excel-файлов.

Обязательные требования:

1. Рассматривай все файлы как взаимосвязанные источники
   по одному процессу.
2. Не структурируй результат по названиям файлов.
3. Не выводи промежуточные результаты.
4. Не упоминай разбиение данных на части.
5. Удали дублирующиеся выводы.
6. Каждый значимый вывод подкрепляй метриками.
7. Отделяй факты от гипотез.
8. Не придумывай причины и показатели.
9. Верни только итоговый отчет в формате,
   установленном основным промптом.

КОНСОЛИДИРОВАННЫЙ МАТЕРИАЛ:

{consolidated_material}
"""

    return cache_call("final", "report", system_prompt, user_prompt, FINAL_MAX_TOKENS)

In [ ]:


if not INPUT_FOLDER.exists():
    INPUT_FOLDER.mkdir(
        parents=True,
        exist_ok=True,
    )

    raise FileNotFoundError(
        f"Создана папка {INPUT_FOLDER.resolve()}. "
        "Помести в нее Excel-файлы и запусти код повторно."
    )


xlsx_files = sorted(
    file_path
    for file_path in INPUT_FOLDER.glob("*.xlsx")
    if not file_path.name.startswith("~$")
)


if not xlsx_files:
    raise FileNotFoundError(
        f"В папке {INPUT_FOLDER.resolve()} "
        "не найдено файлов XLSX."
    )


intermediate_results = []
files_info = []


for file_number, file_path in enumerate(
    xlsx_files,
    start=1,
):
    log(
        f"Обработка файла "
        f"{file_number}/{len(xlsx_files)}: "
        f"{file_path.name}"
    )

    df = pd.read_excel(
        file_path,
        sheet_name=0,
    )

    df = prepare_dataframe(df)

    if df.empty:
        log(
            f"Файл {file_path.name} пустой и пропущен."
        )
        continue

    chunks = split_dataframe_by_size(
        df,
        max_chars=MAX_DATA_CHARS,
    )

    files_info.append({
        "file_name": file_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "parts": len(chunks),
    })

    for part_number, chunk in enumerate(
        tqdm(chunks, desc=f"Части {file_path.name}", unit="часть"),
        start=1,
    ):
        log(
            f"  Часть {part_number}/{len(chunks)}"
        )

        result = analyze_data_chunk(
            file_name=file_path.name,
            chunk_json=chunk["json"],
            part_number=part_number,
            total_parts=len(chunks),
            start_row=chunk["start_row"],
            end_row=chunk["end_row"],
        )

        intermediate_results.append(result)

        if part_number < len(chunks):
            perf_event(stage="analysis", part=part_number, status="fixed_delay", wait_seconds=REQUEST_DELAY_SECONDS)
            time.sleep(REQUEST_DELAY_SECONDS)


if not intermediate_results:
    raise RuntimeError(
        "Не удалось получить материалы "
        "для итогового анализа."
    )


consolidated_material = consolidate_process_materials(
    intermediate_results
)


final_report = create_final_process_report(
    consolidated_material=consolidated_material,
    files_info=files_info,
)


output_path = (
    OUTPUT_FOLDER
    / "process_consolidated_analysis.md"
)


output_path.write_text(
    final_report,
    encoding="utf-8",
)


# Выводится только один итоговый отчет
display(Markdown(final_report))

print(
    f"\nИтоговый отчет сохранен: "
    f"{output_path.resolve()}"
)
# Дополнительно сохраняем HTML и промежуточный консолидированный материал.
(OUTPUT_FOLDER / "process_consolidated_material.md").write_text(consolidated_material, encoding="utf-8")
html_path = OUTPUT_FOLDER / "process_consolidated_analysis.html"
html_path.write_text(render_html(final_report), encoding="utf-8")
print(f"HTML-отчёт: {html_path.resolve()}")
print(f"Журнал производительности: {PERFORMANCE_LOG.resolve()}")


# 